# Task 1 - TinyStories preprocessing (Aswin)
This notebook runs locally or in Colab and discovers the repository automatically. It downloads TinyStories, validates records, creates Aswin's seeded 100,000/10,000 story split, builds a character vocabulary, and saves fixed-length shifted sequences.

In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys

# Colab setup: install only missing packages and locate the uploaded repository automatically.
packages = {'datasets': 'datasets', 'yaml': 'pyyaml', 'psutil': 'psutil', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'tqdm': 'tqdm'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

def locate_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'task1_llm/member_aswin/configs/full.yaml').exists():
            return candidate.resolve()
    content = Path('/content')
    matches = list(content.rglob('task1_llm/member_aswin/configs/full.yaml')) if content.exists() else []
    if not matches:
        try:
            from google.colab import drive
            print('Repository not found in /content; mounting Google Drive ...')
            drive.mount('/content/drive')
            drive_root = Path('/content/drive/MyDrive')
            matches = list(drive_root.rglob('task1_llm/member_aswin/configs/full.yaml'))
        except ImportError:
            pass
    if not matches:
        raise FileNotFoundError('Upload the complete folder containing task1_llm to /content or My Drive, then rerun.')
    return matches[0].resolve().parents[3]

REPO_ROOT = locate_repo_root(); os.chdir(REPO_ROOT)
CONFIG = REPO_ROOT / 'task1_llm/member_aswin/configs/full.yaml'
SCRIPT = REPO_ROOT / 'task1_llm/member_aswin/src/preprocess.py'
required = [SCRIPT, REPO_ROOT / 'task1_llm/member_aswin/src/common.py']
assert all(path.exists() for path in required), f'Missing supporting files under {REPO_ROOT}'
import yaml
RUN_ID = yaml.safe_load(CONFIG.read_text())['run_id']
print('Repository root:', REPO_ROOT)
print('Run ID:', RUN_ID)
subprocess.run([sys.executable, str(SCRIPT), '--config', str(CONFIG)], cwd=REPO_ROOT, check=True)

In [ ]:
import json
metadata_path = REPO_ROOT / 'task1_llm/member_aswin/data_processed' / RUN_ID / 'metadata.json'
metadata = json.loads(metadata_path.read_text())
{k: metadata[k] for k in ['dataset', 'seed', 'context_length', 'train_examples', 'val_examples', 'vocab_size', 'raw_quality']}

In [ ]:
import torch
batch = torch.load(REPO_ROOT / 'task1_llm/member_aswin/data_processed' / RUN_ID / 'train.pt', weights_only=True)['sequences'][:2]
x, y = batch[:, :-1], batch[:, 1:]
assert torch.equal(x[:, 1:], y[:, :-1])
print('Input-target one-character shift verified:', tuple(x.shape), tuple(y.shape))